# Descrizioni dei nodi ed embedding di frasi

Obiettivo: trovare un'entità **descritta senza il suo nome** (tipo R1 del catalogo), per esempio *"Croatian midfielder who won the Golden Ball in 2018"* → Luka Modrić. Il full-text dei nomi (notebook 03) qui non serve, perché il nome non c'è.

Come:
1. **verbalizzazione del grafo**: per ognuno dei 12.893 nodi delle sei classi principali (Player, Participation, Match, Federation, Club, Stadium) si scrive una frase in inglese con una frase-modello riempita con i dati del grafo (codice in `src/wc_retrieval/descrizioni.py`);
2. ogni frase diventa un vettore di 384 numeri con il modello di frasi **`all-MiniLM-L6-v2`**: frasi con significato simile hanno vettori vicini;
3. la domanda passa nello stesso modello e si cercano le descrizioni più vicine, con un **indice vettoriale per classe** in Neo4j (senza quantizzazione e con sovracampionamento, come per le immagini nel notebook 04);
4. il nodo trovato passa al modello Cypher del suo tipo (scheda giocatore, scheda partita…): *la ricerca semantica trova, il grafo risponde*.

Le descrizioni e i vettori sono dati derivati: si salvano in `data/embeddings/descrizioni_minilm.parquet` e si scrivono in Neo4j come proprietà `description` ed `embedding`, non nelle triple RDF. La cella 5 è l'unica che scrive nel database.

**Prerequisiti:** nessuna installazione nuova (`sentence-transformers` è già installato per il notebook 04); al primo avvio il modello (circa 90 MB) si scarica da Hugging Face. **Tempi indicativi:** lettura dei dati e frasi 1–2 minuti, embedding qualche minuto su CPU.

In [ ]:
# [Cella 1] Setup: codice del retrieval, percorsi e connessione a Neo4j
import importlib
import sys
import time
from getpass import getpass
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
import wc_retrieval.descrizioni
import wc_retrieval.embedding
import wc_retrieval.grafo
import wc_retrieval.ricerca
for modulo in (wc_retrieval.grafo, wc_retrieval.ricerca, wc_retrieval.descrizioni, wc_retrieval.embedding):
    importlib.reload(modulo)                              # così una correzione in src/ si prova senza riavviare
from wc_retrieval import descrizioni, ricerca
from wc_retrieval import embedding as emb
from wc_retrieval.grafo import Grafo

EMB = ROOT / "data" / "embeddings"                        # vettori (dati derivati)
EMB.mkdir(parents=True, exist_ok=True)
DIR4 = ROOT / "data" / "raw" / "dataset_4" / "Images" / "Images"

# Connessione a Neo4j: le descrizioni si costruiscono leggendo il grafo
g = Grafo(getpass("Password Neo4j: "))
print(g.leggi("MATCH (n:Resource) RETURN count(n) AS nodi").to_string(index=False))

  nodi
126744


## Descrizioni dei nodi
Le query di `descrizioni.QUERY` leggono i dati **in blocco** (una query per tipo di dato, per esempio tutte le convocazioni insieme, invece di una query per nodo) e in sola lettura; le frasi si compongono poi in Python. Le frasi dicono solo fatti presenti nel grafo: le presenze compaiono solo se dataset 1 le registra per tutti i tornei del giocatore (non prima del 1970 né al Mondiale femminile 1999), i club solo per il 2022 (dataset 3).

In [ ]:
# [Cella 2] Descrizioni di tutti i nodi delle sei classi (sola lettura)
t0 = time.time()
# Una frase in inglese per ogni nodo delle sei classi, scritta con i fatti del grafo
desc = descrizioni.crea_descrizioni(g)
print(f"descrizioni: {len(desc)} in {time.time() - t0:.0f} s")

# Controllo: un nodo descritto per ogni nodo della classe
ATTESI = {"Player": 10401, "Participation": 625, "Match": 1248, "Federation": 88, "Club": 291, "Stadium": 240}
conteggi = desc["classe"].value_counts()
for classe, n in ATTESI.items():
    print(f"{classe:14s} {conteggi.get(classe, 0):6d} (attese {n}) {'ok' if conteggi.get(classe, 0) == n else 'DIVERSO'}")
print("URI ripetuti:", desc["uri"].duplicated().sum(), "| descrizioni vuote:", (desc["descrizione"].str.len() == 0).sum())
parole = desc["descrizione"].str.split().str.len()
print("parole per descrizione:")
print(parole.groupby(desc["classe"]).agg(["mean", "min", "max"]).round(1).to_string())

descrizioni: 12893 in 16 s
Player          10401 (attese 10401) ok
Participation     625 (attese 625) ok
Match            1248 (attese 1248) ok
Federation         88 (attese 88) ok
Club              291 (attese 291) ok
Stadium           240 (attese 240) ok
URI ripetuti: 0 | descrizioni vuote: 0
parole per descrizione:
               mean  min  max
classe                       
Club           22.5   12   72
Federation     41.2   28   82
Match          47.0   30   77
Participation  38.6   29   53
Player         21.4   12   70
Stadium        17.8   15   34


## Modello di frasi ed embedding
`all-MiniLM-L6-v2` legge al massimo 256 *token* (pezzi di parola): una frase più lunga verrebbe troncata e la parte finale non conterebbe. La cella controlla che nessuna descrizione superi il limite. I vettori sono normalizzati (lunghezza 1), quindi il prodotto scalare è la similarità coseno, come per CLIP.

In [ ]:
# [Cella 3] Modello di frasi e lunghezza delle descrizioni in token
t0 = time.time()
# all-MiniLM-L6-v2: frasi → vettori di 384 numeri
minilm = emb.carica_modello_testo()
print(f"modello {emb.MODELLO_TESTO} caricato in {time.time() - t0:.0f} s | dimensioni: {minilm.get_sentence_embedding_dimension()}")
# Il modello legge al massimo 256 token: si controlla che nessuna frase venga troncata
token = desc["descrizione"].map(lambda s: len(minilm.tokenizer(s)["input_ids"]))
print(f"limite del modello: {minilm.max_seq_length} token | descrizione più lunga: {token.max()} token "
      f"| troncate: {(token > minilm.max_seq_length).sum()}")
print("token per classe (massimo):", token.groupby(desc["classe"]).max().to_dict())

modello all-MiniLM-L6-v2 caricato in 15 s | dimensioni: 384
limite del modello: 256 token | descrizione più lunga: 163 token | troncate: 0
token per classe (massimo): {'Club': 163, 'Federation': 114, 'Match': 160, 'Participation': 95, 'Player': 109, 'Stadium': 50}


In [ ]:
# [Cella 4] Embedding di tutte le descrizioni e salvataggio
t0 = time.time()
# Vettori normalizzati: il prodotto scalare è la similarità coseno
vettori = minilm.encode(desc["descrizione"].tolist(), batch_size=64, convert_to_numpy=True,
                        normalize_embeddings=True, show_progress_bar=False)
print(f"embedding: {vettori.shape} in {time.time() - t0:.0f} s | norma dei primi tre: {np.linalg.norm(vettori[:3], axis=1).round(3)}")
desc["embedding"] = list(vettori.astype("float32"))
desc.to_parquet(EMB / "descrizioni_minilm.parquet", index=False)
print("salvato:", EMB / "descrizioni_minilm.parquet")

embedding: (12893, 384) in 103 s | norma dei primi tre: [1. 1. 1.]
salvato: <progetto>\data\embeddings\descrizioni_minilm.parquet


## Caricamento in Neo4j
Scrive `description` ed `embedding` sui 12.893 nodi e crea sei indici vettoriali, uno per classe (`player_text`, `participation_text`, `match_text`, `federation_text`, `club_text`, `stadium_text`), con similarità coseno e **senza quantizzazione**. Ogni indice viene cancellato e ricreato. Se il grafo viene ricostruito basta rieseguire 1 e questa cella: legge i vettori dal Parquet.

In [ ]:
# [Cella 5] Caricamento in Neo4j (SCRIVE): description, embedding, indici vettoriali
salvate = pd.read_parquet(EMB / "descrizioni_minilm.parquet")    # si rilegge dal file: funziona anche dopo un riavvio
t0 = time.time()
# Scrive description ed embedding sui 12.893 nodi
print("nodi aggiornati:", emb.carica_descrizioni(g, salvate), f"in {time.time() - t0:.0f} s")
# Un indice vettoriale per classe (player_text, match_text, …)
for classe, indice in ricerca.INDICI_DESCRIZIONI.items():
    emb.crea_indice_vettoriale(g, indice, classe, "embedding", emb.DIMENSIONI_TESTO)
print(g.leggi("""
    SHOW VECTOR INDEXES YIELD name, state, labelsOrTypes, options
    RETURN name AS indice, state AS stato, labelsOrTypes AS classe,
           options.indexConfig['vector.dimensions'] AS dimensioni,
           options.indexConfig['vector.similarity_function'] AS similarita,
           options.indexConfig['vector.quantization.enabled'] AS quantizzazione
    ORDER BY indice""").to_string(index=False))
# per ogni classe: tutti i nodi devono avere descrizione ed embedding
print(pd.concat([g.leggi(f"""
    MATCH (n:{classe})
    RETURN '{classe}' AS classe, count(n) AS nodi, count(n.description) AS con_descrizione,
           count(n.embedding) AS con_embedding""") for classe in ricerca.INDICI_DESCRIZIONI]).to_string(index=False))

nodi aggiornati: 12893 in 28 s
            indice  stato          classe  dimensioni similarita  quantizzazione
         club_text ONLINE          [Club]         384     COSINE           False
   federation_text ONLINE    [Federation]         384     COSINE           False
        image_clip ONLINE         [Image]         512     COSINE           False
        match_text ONLINE         [Match]         384     COSINE           False
participation_text ONLINE [Participation]         384     COSINE           False
       player_text ONLINE        [Player]         384     COSINE           False
      stadium_text ONLINE       [Stadium]         384     COSINE           False
       classe  nodi  con_descrizione  con_embedding
       Player 10401            10401          10401
Participation   625              625            625
        Match  1248             1248           1248
   Federation    88               88             88
         Club   291              291            291
      Sta